# Set 11 – Neuronales Netz für Regression

Wir trainieren mit `MLPRegressor` ein kleines **Multi-Layer Perceptron (MLP)** auf kontrollierten nichtlinearen Daten. Das Netz besitzt genau **ein Output-Neuron**, weil pro Beobachtung ein kontinuierlicher Wert vorhergesagt wird.

## Lernziele

- Schichten und Neuronen mit `hidden_layer_sizes` definieren
- Hidden-Aktivierung und Output-Aktivierung unterscheiden
- Loss, Backpropagation und Optimierer einordnen
- Lernrate, Epochen, Batch-Größe, Early Stopping und L2-Regularisierung verstehen
- eine kleine Hyperparametersuche inklusive Netzstruktur durchführen

> scikit-learn ist für kleine tabellarische MLPs gut geeignet. Es bietet aber keine frei zusammensetzbaren Layer, kein Dropout und keine unterschiedlichen Aktivierungen pro Hidden Layer. Dafür wären Frameworks wie PyTorch oder TensorFlow gedacht.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Kontrollierte Regressionsdaten

Die Zielvariable besteht aus einer Sinuskurve, einem linearen Trend und Rauschen. Mit nur einem Inputmerkmal können wir die gelernte Funktion direkt zeichnen.

- **Input-Schicht:** ein Merkmal $x$
- **Hidden Layer:** lernen nichtlineare Zwischenrepräsentationen
- **Output-Schicht:** genau ein Neuron für einen kontinuierlichen Wert


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X = np.sort(rng.uniform(-3.5, 3.5, 500)).reshape(-1, 1)
y_clean = 1.3 * np.sin(1.4 * X[:, 0]) + 0.25 * X[:, 0]
y = y_clean + rng.normal(0, 0.22, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], y_train, s=24, alpha=0.65, label="Training")
plt.scatter(X_test[:, 0], y_test, s=28, alpha=0.75, label="Test")
plt.xlabel("x"); plt.ylabel("y"); plt.title("Nichtlineare Regressionsdaten")
plt.legend(); plt.show()


## 2. Architektur und Output-Schicht

`hidden_layer_sizes=(32, 16)` bedeutet:

- erster Hidden Layer: 32 Neuronen
- zweiter Hidden Layer: 16 Neuronen
- Output: wird von scikit-learn automatisch erzeugt

Für Regression verwendet `MLPRegressor` am Output die **Identity-Aktivierung** $f(z)=z$. Der Wert wird dadurch nicht auf ein Intervall eingeschränkt. Ein Sigmoid-Output wäre hier ungeeignet, weil er nur Werte zwischen 0 und 1 liefern könnte.

Der interne Loss ist der quadrierte Fehler. scikit-learn optimiert diesen Trainings-Loss; für die verständliche Modellbewertung berichten wir zusätzlich MAE, RMSE und $R^2$.


In [ ]:
def draw_network(layer_sizes, labels):
    # Kleine schematische Darstellung; keine exakten trainierten Gewichte.
    fig, ax = plt.subplots(figsize=(10, 5))
    x_positions = np.linspace(0, 1, len(layer_sizes))
    coordinates = []
    for x_pos, size in zip(x_positions, layer_sizes):
        shown = min(size, 8)  # große Layer nur symbolisch darstellen
        ys = np.linspace(0.12, 0.88, shown)
        coordinates.append([(x_pos, y_pos) for y_pos in ys])
    for left, right in zip(coordinates[:-1], coordinates[1:]):
        for x1, y1 in left:
            for x2, y2 in right:
                ax.plot([x1, x2], [y1, y2], color="lightgrey", lw=0.5, zorder=1)
    for layer, label, size in zip(coordinates, labels, layer_sizes):
        for x_pos, y_pos in layer:
            ax.scatter(x_pos, y_pos, s=260, color="#4C78A8", edgecolor="white", zorder=2)
        ax.text(layer[0][0], 0.02, f"{label}\n({size})", ha="center", va="top")
    ax.set(xlim=(-0.1, 1.1), ylim=(-0.08, 1), title="MLP-Architektur (schematisch)")
    ax.axis("off"); plt.show()

draw_network([1, 32, 16, 1], ["Input", "Hidden 1", "Hidden 2", "Output linear"])


## 3. Modellparameter verständlich lesen

| Parameter | Rolle |
|---|---|
| `hidden_layer_sizes` | Anzahl der Neuronen pro Hidden Layer |
| `activation` | Aktivierung aller Hidden Layer: `relu`, `tanh`, `logistic` oder `identity` |
| `solver` | Optimierer: `adam`, `sgd` oder für kleine Daten `lbfgs` |
| `learning_rate_init` | anfängliche Schrittweite der Gewichtsupdates |
| `alpha` | Stärke der L2-Regularisierung; große Gewichte werden bestraft |
| `batch_size` | Zahl der Trainingsbeispiele pro Update |
| `max_iter` | maximale Zahl der Epochen bei Adam/SGD |
| `early_stopping` | beendet das Training, wenn der Validierungsscore nicht besser wird |
| `n_iter_no_change` | Geduld für Early Stopping |

**Adam** kombiniert adaptive Schrittweiten mit einer Art Momentum. Eine zu große Lernrate kann am Minimum vorbeispringen; eine zu kleine macht das Training langsam. `alpha` ist bei scikit-learn L2-Regularisierung – nicht L1 und nicht Dropout.


In [ ]:
regressor = Pipeline([
    # Der Scaler lernt nur aus Trainingsdaten und transformiert danach automatisch.
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(32, 16),  # zwei kleine Hidden Layer
        activation="relu",           # nichtlineare Aktivierung in Hidden Layern
        solver="adam",               # Optimierer
        learning_rate_init=0.01,      # anfängliche Lernrate
        alpha=0.001,                  # L2-Regularisierung
        batch_size=32,                # Beobachtungen pro Gewichtsupdate
        max_iter=1200,                # maximale Zahl der Epochen
        early_stopping=True,          # internes Validierungsset überwachen
        validation_fraction=0.15,
        n_iter_no_change=40,
        random_state=RANDOM_STATE,
    )),
])

regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

def regression_metrics(y_true, prediction):
    return {
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_true, prediction)),
        "R2": r2_score(y_true, prediction),
    }

print({name: round(value, 3) for name, value in regression_metrics(y_test, y_pred).items()})


## 4. Was hat das Netz tatsächlich aufgebaut?

Nach `fit` stellt scikit-learn gelernte Eigenschaften bereit:

- `n_layers_`: Input-, Hidden- und Output-Schichten zusammen
- `n_outputs_`: Zahl der vorhergesagten Zielwerte
- `out_activation_`: automatisch gewählte Output-Aktivierung
- `coefs_`: Gewichtsmatrizen zwischen den Schichten
- `intercepts_`: Bias-Vektoren
- `loss_curve_`: Trainings-Loss nach jeder Epoche
- `n_iter_`: tatsächlich durchlaufene Epochen


In [ ]:
mlp = regressor.named_steps["mlp"]
print("Schichten einschließlich Input/Output:", mlp.n_layers_)
print("Output-Neuronen:", mlp.n_outputs_)
print("Output-Aktivierung:", mlp.out_activation_)
print("Epochen:", mlp.n_iter_)
for index, weights in enumerate(mlp.coefs_, 1):
    print(f"Gewichtsmatrix {index}: {weights.shape}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(mlp.loss_curve_)
axes[0].set(title="Trainings-Loss", xlabel="Epoche", ylabel="Loss")

x_grid = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
axes[1].scatter(X_train[:, 0], y_train, s=18, alpha=0.45, label="Training")
axes[1].plot(x_grid[:, 0], regressor.predict(x_grid), color="crimson", lw=2.5, label="MLP")
axes[1].set(title="Gelernte Regressionsfunktion", xlabel="x", ylabel="y")
axes[1].legend(); plt.tight_layout(); plt.show()


## 5. Kleine Hyperparametersuche

Wir verändern die Struktur nur moderat. Zu große Netze wären für diesen kleinen Datensatz unnötig und würden die Suche verlangsamen.

Der Präfix `mlp__` adressiert Parameter des Pipeline-Schritts. Die Auswahl erfolgt per 3-Fold-Cross-Validation auf den Trainingsdaten. Der Testdatensatz bleibt bis zur Schlussbewertung unberührt.


In [ ]:
search_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        solver="adam",
        batch_size=32,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=30,
        random_state=RANDOM_STATE,
    )),
])

param_grid = {
    "mlp__hidden_layer_sizes": [(16,), (32,), (32, 16)],
    "mlp__activation": ["relu", "tanh"],
    "mlp__learning_rate_init": [0.001, 0.01],
    "mlp__alpha": [0.0001, 0.01],
}

search = GridSearchCV(
    search_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    return_train_score=True,
)
search.fit(X_train, y_train)
print("Beste Parameter:", search.best_params_)
print("Bester CV-RMSE:", round(-search.best_score_, 3))


In [ ]:
results = pd.DataFrame(search.cv_results_)
display(results[["params", "mean_train_score", "mean_test_score", "std_test_score"]]
        .sort_values("mean_test_score", ascending=False).head(8).round(3))

best_regressor = search.best_estimator_
best_prediction = best_regressor.predict(X_test)
print("Finale Testmetriken:",
      {name: round(value, 3) for name, value in regression_metrics(y_test, best_prediction).items()})


## Fazit

- Regression benötigt hier genau ein lineares Output-Neuron.
- Nichtlinearität entsteht in den Hidden Layern durch ReLU oder tanh.
- Backpropagation berechnet Gradienten; Adam aktualisiert damit Gewichte und Biases.
- Lernrate, Netzgröße und Epochen bestimmen, ob das Modell ausreichend und stabil lernt.
- L2-Regularisierung und Early Stopping wirken Overfitting entgegen.
- Hyperparameter werden auf Trainingsdaten per Cross-Validation gewählt.
